# Housing Price Regression Analysis

**Internship:** XYlofy — Week 1 Assignment  
**Author:** Eakansh Kshirsagar  
**Dataset:** `Housing.csv` (545 rows × 13 columns)  

This notebook completes all 5 required tasks:
1. Data Loading & Exploration
2. Data Cleaning
3. Model Building (Linear Regression + Random Forest)
4. Visualization (3 charts)
5. Insights & Summary

In [ ]:
# Install all required libraries (safe to re-run)
!pip install pandas numpy matplotlib seaborn scikit-learn --quiet

In [ ]:
# Import libraries
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving charts
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Suppress deprecation warnings for clean output
warnings.filterwarnings('ignore')

# Set aesthetic style and reproducibility seed
sns.set_style('whitegrid')
np.random.seed(42)

print('All libraries loaded successfully!')

---
## Task 1 — Data Loading & Exploration

In [ ]:
# Load the CSV file (relative path — works on any machine)
df = pd.read_csv('Housing.csv')

# Display the first 10 rows
print('--- First 10 Rows ---')
df.head(10)

In [ ]:
# Check how many rows and columns
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

# Identify the target (price) and feature columns
target_column = 'price'
feature_columns = [col for col in df.columns if col != target_column]
print(f'\nTarget column: {target_column}')
print(f'Feature columns ({len(feature_columns)}): {feature_columns}')

In [ ]:
# Check for missing values in each column
print('--- Missing Values per Column ---')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

---
## Task 2 — Data Cleaning

In [ ]:
# 1. Handle missing values
# Numerical columns — fill with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_column in num_cols:
    num_cols.remove(target_column)
for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  Filled {col} missing values with median ({median_val})')

# Categorical columns — fill with mode
cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  Filled {col} missing values with mode ({mode_val})')

if df.isnull().sum().sum() == 0:
    print('No missing values found (or all handled).')

In [ ]:
# 2. Remove duplicate rows
dups = df.duplicated().sum()
if dups > 0:
    df = df.drop_duplicates()
    print(f'Removed {dups} duplicate rows.')
else:
    print('No duplicate rows found.')

print(f'Dataset shape after cleaning: {df.shape}')

In [ ]:
# 3. One-hot encode categorical columns (yes/no fields → numeric)
print(f'Categorical columns to encode: {cat_cols}')
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert boolean columns to int (0/1) for compatibility
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'\nDataset shape after encoding: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## Task 3 — Model Building

In [ ]:
# Prepare features (X) and target (y)
X = df.drop(columns=[target_column])
y = df[target_column]

# Split the data into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')

In [ ]:
# Train Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Train Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('Both models trained successfully!')

In [ ]:
# Evaluate both models using MAE, RMSE, and R² Score
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R² Score': round(r2, 3)}

results = pd.DataFrame([
    evaluate_model('Linear Regression', y_test, y_pred_lr),
    evaluate_model('Random Forest',     y_test, y_pred_rf)
])

print('--- Model Comparison ---')
results

---
## Task 4 — Visualization (3 Charts)

In [ ]:
# Create charts folder to save PNG images
os.makedirs('charts', exist_ok=True)

# ── Chart 1: Histogram showing the distribution of house prices ──
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df[target_column], kde=True, bins=30, color='steelblue', ax=ax)
ax.set_title('Distribution of House Prices')
ax.set_xlabel('Price')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig('charts/price_histogram.png', dpi=150)
plt.show()
print('Saved: charts/price_histogram.png')

In [ ]:
# ── Chart 2: Correlation heatmap ──
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='viridis', ax=ax)
ax.set_title('Feature Correlation Heatmap')
fig.tight_layout()
fig.savefig('charts/correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: charts/correlation_heatmap.png')

In [ ]:
# ── Chart 3: Actual vs Predicted Prices (Random Forest) ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred_rf, alpha=0.7, edgecolors='k', linewidths=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Price')
ax.set_ylabel('Predicted Price')
ax.set_title('Actual vs Predicted Prices (Random Forest)')
fig.tight_layout()
fig.savefig('charts/actual_vs_predicted.png', dpi=150)
plt.show()
print('Saved: charts/actual_vs_predicted.png')

---
## Task 5 — Insights & Summary

In [ ]:
# Feature importance from Random Forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print('--- Top 10 Feature Importances (Random Forest) ---')
importances.head(10)

### Key Findings

1. **Most Influential Features** — Based on the Random Forest feature importances, the top drivers of house price are **area** (46.9%), **bathrooms** (15.3%), and **airconditioning** (6.0%). The total area of a property is by far the strongest predictor of its price.

2. **Model Accuracy** — The Linear Regression model achieved a MAE of **970,043** and an R² of **0.653**, while the Random Forest achieved a MAE of **1,013,969** and an R² of **0.613**. Both models explain roughly **61–65%** of the variance in house prices, providing a solid baseline for price estimation.

3. **What Surprised Me** — The feature **area** dominates the prediction far more than the number of bedrooms or bathrooms. This suggests that buyers value raw living space much more than how many rooms a house is divided into. Also, air conditioning presence had a surprisingly strong effect on price — stronger than parking, stories, or even having a basement.

4. **Recommendation** — Real estate businesses should prioritize accurate area measurements and air conditioning availability when pricing properties. Investing in AC installation and marketing spacious layouts could significantly increase a property’s perceived and actual market value.